# 🦅 Set Up

In [1]:
import json
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

pd.set_option('display.max_columns', None)

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms

import requests
from PIL import Image
from io import BytesIO
from torch.autograd import Variable
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [12]:
def get_vector_fromURL(url):
    response = requests.get(url)
    img = Image.open(BytesIO(response.content)).convert('RGB').resize((300, 300))
    t_img = Variable(normalize(to_tensor(scaler(img))).unsqueeze(0))
    my_embedding = torch.zeros(512)
    def copy_data(m, i, o):
        my_embedding.copy_(o.data.reshape(o.data.size(1)))
    h = layer.register_forward_hook(copy_data)
    model(t_img)
    h.remove()
    return my_embedding

In [31]:
def save_as_dataframe(dict_):
    df = pd.DataFrame(dict_).T
    df.columns = ['ResNet'+str(i) for i in range(1, 513)]
    return df

In [4]:
cd ..

C:\Users\chopi\Penn Dropbox\Hyunwoo Jung\1_Personal\_Hyunwoo Place\graduate school\2_coursework (2025-F)\2_CIS5200_Machine Learning\5_final project\3_analyses


# 🦅 Load Data

In [8]:
df_reviews = pd.read_feather('data/appliances_meta_012023_062023 v1.1.0.ftr')

In [10]:
df_reviews.head()

,review_id,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,medium_image_url,medium_image_url_1,medium_image_url_2,medium_image_url_3,medium_image_url_4,medium_image_url_5,medium_image_url_6,medium_image_url_7,medium_image_url_8,medium_image_url_9,medium_image_url_10,medium_image_url_11,medium_image_url_12,medium_image_url_13,medium_image_url_14,medium_image_url_15,medium_image_url_16,medium_image_url_17,medium_image_url_18,medium_image_url_19,medium_image_url_20,medium_image_url_21,medium_image_url_22,medium_image_url_23,medium_image_url_24,medium_image_url_25,medium_image_url_26,medium_image_url_27,text_len,text_words,inter_review_time
0,1,3.0,Needs hose clamps,Needs metal hose clamps not plastic ties.... C...,B00004YWK2,B00004YWK2,AFFPAJDCW7NSKE4FZWBRWETUKZ2A,2023-02-18 02:32:06.278,0,True,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,108,18,NaT
1,2,1.0,Don't waste your money,Product is cheap and the door doesn't work well,B00004YWK2,B00004YWK2,AEI6B25VF65CG2HPBQ2FNBG7IQKA,2023-02-10 00:31:39.499,0,True,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,47,9,8 days 02:00:26.779000
2,3,1.0,Must have been a return,No cover to close it and plastic was separated.,B00004YWK2,B00004YWK2,AGA5X6NUWSQM42KDFJLO25JBXOEA,2023-02-07 14:55:58.766,0,True,https://m.media-amazon.com/images/I/71eRK2xYuJ...,https://m.media-amazon.com/images/I/71eRK2xYuJ...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,47,9,2 days 09:35:40.733000
3,4,5.0,Great product,"I love this it keeps my garage, nice and warm ...",B00004YWK2,B00004YWK2,AGO4SBTXOUTKYMHKQQNX7ZFDQSFA,2023-01-29 19:37:40.587,0,True,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,115,24,8 days 19:18:18.179000
4,5,5.0,Why be wasteful?,"So I mounted this, as you see, behind and just...",B00004YWK2,B00004YWK2,AHKTIX6L7FKPYDENUALEPNPMAIHQ,2023-01-03 16:32:51.697,0,True,https://m.media-amazon.com/images/I/61ctMn42LX...,https://m.media-amazon.com/images/I/61ctMn42LX...,https://m.media-amazon.com/images/I/71QP-FAvlQ...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,708,148,26 days 03:04:48.890000


## 🐔 get urls

In [14]:
df_reviews_imgs = df_reviews[['review_id'] +[col for col in df_reviews.columns if 'medium_image_url' in col]]
df_reviews_imgs = df_reviews_imgs[df_reviews_imgs['medium_image_url'].notnull()]

In [19]:
df_reviews_imgs.head()

,review_id,medium_image_url,medium_image_url_1,medium_image_url_2,medium_image_url_3,medium_image_url_4,medium_image_url_5,medium_image_url_6,medium_image_url_7,medium_image_url_8,medium_image_url_9,medium_image_url_10,medium_image_url_11,medium_image_url_12,medium_image_url_13,medium_image_url_14,medium_image_url_15,medium_image_url_16,medium_image_url_17,medium_image_url_18,medium_image_url_19,medium_image_url_20,medium_image_url_21,medium_image_url_22,medium_image_url_23,medium_image_url_24,medium_image_url_25,medium_image_url_26,medium_image_url_27
2,3,https://m.media-amazon.com/images/I/71eRK2xYuJ...,https://m.media-amazon.com/images/I/71eRK2xYuJ...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
4,5,https://m.media-amazon.com/images/I/61ctMn42LX...,https://m.media-amazon.com/images/I/61ctMn42LX...,https://m.media-amazon.com/images/I/71QP-FAvlQ...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
16,17,https://m.media-amazon.com/images/I/61VedqsVak...,https://m.media-amazon.com/images/I/61VedqsVak...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
50,51,https://m.media-amazon.com/images/I/71S7C6wu2h...,https://m.media-amazon.com/images/I/71S7C6wu2h...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
102,103,https://m.media-amazon.com/images/I/61vLTOZo2L...,https://m.media-amazon.com/images/I/61vLTOZo2L...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None


In [16]:
url_cols = [col for col in df_reviews_imgs.columns if 'medium_image_url' in col]

# Melt the dataframe to long format
df_reviews_imgs_long = df_reviews_imgs.melt(
    id_vars=['review_id'],
    value_vars=url_cols,
    var_name='image_slot',
    value_name='image_url'
)

# Drop rows with missing URLs
df_reviews_imgs_long = df_reviews_imgs_long.dropna(subset=['image_url'])

# Keep only the relevant columns
df_reviews_imgs_long = df_reviews_imgs_long[['review_id', 'image_url']].sort_values('review_id').reset_index(drop=True)

In [20]:
df_reviews_imgs_long.head(10)

,review_id,image_url
0,3,https://m.media-amazon.com/images/I/71eRK2xYuJ...
1,3,https://m.media-amazon.com/images/I/71eRK2xYuJ...
2,5,https://m.media-amazon.com/images/I/61ctMn42LX...
3,5,https://m.media-amazon.com/images/I/61ctMn42LX...
4,5,https://m.media-amazon.com/images/I/71QP-FAvlQ...
5,17,https://m.media-amazon.com/images/I/61VedqsVak...
6,17,https://m.media-amazon.com/images/I/61VedqsVak...
7,51,https://m.media-amazon.com/images/I/71S7C6wu2h...
8,51,https://m.media-amazon.com/images/I/71S7C6wu2h...
9,103,https://m.media-amazon.com/images/I/61vLTOZo2L...


In [21]:
df_reviews_imgs_long.to_feather('data/appliances_meta_012023_062023 v1.1.0_imgUrls.ftr')

# 🦅 Extract ResNet Vector

## 🐔 extract embeddings

In [5]:
df_photoURL = pd.read_feather('data/appliances_meta_012023_062023 v1.1.0_imgUrls.ftr')

In [6]:
df_photoURL.head()

,review_id,image_url
0,3,https://m.media-amazon.com/images/I/71eRK2xYuJ...
1,3,https://m.media-amazon.com/images/I/71eRK2xYuJ...
2,5,https://m.media-amazon.com/images/I/61ctMn42LX...
3,5,https://m.media-amazon.com/images/I/61ctMn42LX...
4,5,https://m.media-amazon.com/images/I/71QP-FAvlQ...


In [7]:
len(df_photoURL)

24684

In [8]:
''' model setting '''
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(pretrained=True) 
layer = model._modules.get('avgpool')    
model.eval()                          

C:\Users\chopi\anaconda3\envs\ml-project\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\chopi\anaconda3\envs\ml-project\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [9]:
''' prep image data '''
scaler = transforms.Resize((224, 224))
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
to_tensor = transforms.ToTensor()

In [37]:
''' change photo to vector '''
dict_data = {}
list_failed_URL = []
for idx in tqdm(df_photoURL.index):
    url = df_photoURL.loc[idx, 'image_url']
    try: 
        dict_data[url] = np.array(get_vector_fromURL(url))
    except Exception as e:
        print("index {}: with Error {}".format(idx, e))
        list_failed_URL.append({'image_url': url, 'idx': idx})

  0%|          | 0/24684 [00:00<?, ?it/s]

index 40: with Error cannot identify image file <_io.BytesIO object at 0x000001610BFDBF90>
index 41: with Error cannot identify image file <_io.BytesIO object at 0x00000161115A22C0>
index 998: with Error cannot identify image file <_io.BytesIO object at 0x000001611029CE50>
index 999: with Error cannot identify image file <_io.BytesIO object at 0x00000161113B3590>
index 2021: with Error cannot identify image file <_io.BytesIO object at 0x0000016111560450>
index 2022: with Error cannot identify image file <_io.BytesIO object at 0x00000161115F4680>
index 2084: with Error cannot identify image file <_io.BytesIO object at 0x000001611153FB80>
index 2085: with Error cannot identify image file <_io.BytesIO object at 0x000001611160FA40>
index 2491: with Error cannot identify image file <_io.BytesIO object at 0x000001611153F810>
index 2492: with Error cannot identify image file <_io.BytesIO object at 0x0000016109902BD0>
index 2493: with Error cannot identify image file <_io.BytesIO object at 0x0

In [38]:
len(list_failed_URL)

131

In [39]:
df_imgEmb = save_as_dataframe(dict_data).reset_index().rename(columns={'index':'image_url'})

In [41]:
df_imgEmb['image_url'].nunique()

15566

In [45]:
df_imgEmb.shape, df_photoURL.shape

((15566, 513), (24684, 2))

In [42]:
df_imgEmb.head()

,image_url,ResNet1,ResNet2,ResNet3,ResNet4,ResNet5,ResNet6,ResNet7,ResNet8,ResNet9,ResNet10,ResNet11,ResNet12,ResNet13,ResNet14,ResNet15,ResNet16,ResNet17,ResNet18,ResNet19,ResNet20,ResNet21,ResNet22,ResNet23,ResNet24,ResNet25,ResNet26,ResNet27,ResNet28,ResNet29,ResNet30,ResNet31,ResNet32,ResNet33,ResNet34,ResNet35,ResNet36,ResNet37,ResNet38,ResNet39,ResNet40,ResNet41,ResNet42,ResNet43,ResNet44,ResNet45,ResNet46,ResNet47,ResNet48,ResNet49,ResNet50,ResNet51,ResNet52,ResNet53,ResNet54,ResNet55,ResNet56,ResNet57,ResNet58,ResNet59,ResNet60,ResNet61,ResNet62,ResNet63,ResNet64,ResNet65,ResNet66,ResNet67,ResNet68,ResNet69,ResNet70,ResNet71,ResNet72,ResNet73,ResNet74,ResNet75,ResNet76,ResNet77,ResNet78,ResNet79,ResNet80,ResNet81,ResNet82,ResNet83,ResNet84,ResNet85,ResNet86,ResNet87,ResNet88,ResNet89,ResNet90,ResNet91,ResNet92,ResNet93,ResNet94,ResNet95,ResNet96,ResNet97,ResNet98,ResNet99,ResNet100,ResNet101,ResNet102,ResNet103,ResNet104,ResNet105,ResNet106,ResNet107,ResNet108,ResNet109,ResNet110,ResNet111,ResNet112,ResNet113,ResNet114,ResNet115,ResNet116,ResNet117,ResNet118,ResNet119,ResNet120,ResNet121,ResNet122,ResNet123,ResNet124,ResNet125,ResNet126,ResNet127,ResNet128,ResNet129,ResNet130,ResNet131,ResNet132,ResNet133,ResNet134,ResNet135,ResNet136,ResNet137,ResNet138,ResNet139,ResNet140,ResNet141,ResNet142,ResNet143,ResNet144,ResNet145,ResNet146,ResNet147,ResNet148,ResNet149,ResNet150,ResNet151,ResNet152,ResNet153,ResNet154,ResNet155,ResNet156,ResNet157,ResNet158,ResNet159,ResNet160,ResNet161,ResNet162,ResNet163,ResNet164,ResNet165,ResNet166,ResNet167,ResNet168,ResNet169,ResNet170,ResNet171,ResNet172,ResNet173,ResNet174,ResNet175,ResNet176,ResNet177,ResNet178,ResNet179,ResNet180,ResNet181,ResNet182,ResNet183,ResNet184,ResNet185,ResNet186,ResNet187,ResNet188,ResNet189,ResNet190,ResNet191,ResNet192,ResNet193,ResNet194,ResNet195,ResNet196,ResNet197,ResNet198,ResNet199,ResNet200,ResNet201,ResNet202,ResNet203,ResNet204,ResNet205,ResNet206,ResNet207,ResNet208,ResNet209,ResNet210,ResNet211,ResNet212,ResNet213,ResNet214,ResNet215,ResNet216,ResNet217,ResNet218,ResNet219,ResNet220,ResNet221,ResNet222,ResNet223,ResNet224,ResNet225,ResNet226,ResNet227,ResNet228,ResNet229,ResNet230,ResNet231,ResNet232,ResNet233,ResNet234,ResNet235,ResNet236,ResNet237,ResNet238,ResNet239,ResNet240,ResNet241,ResNet242,ResNet243,ResNet244,ResNet245,ResNet246,ResNet247,ResNet248,ResNet249,ResNet250,ResNet251,ResNet252,ResNet253,ResNet254,ResNet255,ResNet256,ResNet257,ResNet258,ResNet259,ResNet260,ResNet261,ResNet262,ResNet263,ResNet264,ResNet265,ResNet266,ResNet267,ResNet268,ResNet269,ResNet270,ResNet271,ResNet272,ResNet273,ResNet274,ResNet275,ResNet276,ResNet277,ResNet278,ResNet279,ResNet280,ResNet281,ResNet282,ResNet283,ResNet284,ResNet285,ResNet286,ResNet287,ResNet288,ResNet289,ResNet290,ResNet291,ResNet292,ResNet293,ResNet294,ResNet295,ResNet296,ResNet297,ResNet298,ResNet299,ResNet300,ResNet301,ResNet302,ResNet303,ResNet304,ResNet305,ResNet306,ResNet307,ResNet308,ResNet309,ResNet310,ResNet311,ResNet312,ResNet313,ResNet314,ResNet315,ResNet316,ResNet317,ResNet318,ResNet319,ResNet320,ResNet321,ResNet322,ResNet323,ResNet324,ResNet325,ResNet326,ResNet327,ResNet328,ResNet329,ResNet330,ResNet331,ResNet332,ResNet333,ResNet334,ResNet335,ResNet336,ResNet337,ResNet338,ResNet339,ResNet340,ResNet341,ResNet342,ResNet343,ResNet344,ResNet345,ResNet346,ResNet347,ResNet348,ResNet349,ResNet350,ResNet351,ResNet352,ResNet353,ResNet354,ResNet355,ResNet356,ResNet357,ResNet358,ResNet359,ResNet360,ResNet361,ResNet362,ResNet363,ResNet364,ResNet365,ResNet366,ResNet367,ResNet368,ResNet369,ResNet370,ResNet371,ResNet372,ResNet373,ResNet374,ResNet375,ResNet376,ResNet377,ResNet378,ResNet379,ResNet380,ResNet381,ResNet382,ResNet383,ResNet384,ResNet385,ResNet386,ResNet387,ResNet388,ResNet389,ResNet390,ResNet391,ResNet392,ResNet393,ResNet394,ResNet395,ResNet396,ResNet397,ResNet398,ResNet399,ResNet400,ResNet401,ResNet402,ResNet403,ResNet404,ResNet405,ResNet406,ResNet407,ResNet408,ResNet409,ResNet4

In [43]:
df_imgEmb.to_csv('data/appliances_meta_012023_062023 v1.1.0_imgUrls_imgEmb.csv', index=False)

## 🐔 debug error lines

In [50]:
df_photoURL = pd.read_feather('data/appliances_meta_012023_062023 v1.1.0_imgUrls.ftr').drop_duplicates(subset='image_url')
df_photoURL_imgEmb = pd.read_csv('data/appliances_meta_012023_062023 v1.1.0_imgUrls_imgEmb.csv')

In [51]:
df_photoURL_leftovers = df_photoURL[~df_photoURL['image_url'].isin(df_photoURL_imgEmb['image_url'].unique())]

In [53]:
df_photoURL['image_url'].nunique(), df_photoURL_leftovers['image_url'].nunique(), df_photoURL_imgEmb['image_url'].nunique()

(15651, 85, 15566)

In [54]:
len(df_photoURL_leftovers)

85

In [55]:
df_photoURL_leftovers.head()

,review_id,image_url
40,285,https://m.media-amazon.com/images/I/71Wr5Hq-RD...
998,7100,https://m.media-amazon.com/images/I/611p7No9iZ...
2021,14261,https://m.media-amazon.com/images/I/81HU8d+FTS...
2084,14936,https://m.media-amazon.com/images/I/71kLb368OQ...
2491,17822,https://m.media-amazon.com/images/I/71fw2n5NAe...


In [57]:
''' model setting '''
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(pretrained=True) 
layer = model._modules.get('avgpool')    
model.eval()                          

C:\Users\chopi\anaconda3\envs\ml-project\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\chopi\anaconda3\envs\ml-project\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [58]:
''' prep image data '''
scaler = transforms.Resize((224, 224))
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
to_tensor = transforms.ToTensor()

In [59]:
''' change photo to vector '''
dict_data = {}
list_failed_URL = []
for idx in tqdm(df_photoURL_leftovers.index):
    url = df_photoURL_leftovers.loc[idx, 'image_url']
    try: 
        dict_data[url] = np.array(get_vector_fromURL(url))
    except Exception as e:
        print("index {}: with Error {}".format(idx, e))
        list_failed_URL.append({'image_url': url, 'idx': idx})

  0%|          | 0/85 [00:00<?, ?it/s]

index 40: with Error cannot identify image file <_io.BytesIO object at 0x0000016111BF7310>
index 998: with Error cannot identify image file <_io.BytesIO object at 0x0000016111BF79A0>
index 2021: with Error cannot identify image file <_io.BytesIO object at 0x000001611154F540>
index 2084: with Error cannot identify image file <_io.BytesIO object at 0x000001611153F810>
index 2491: with Error cannot identify image file <_io.BytesIO object at 0x00000161113B6C70>
index 2493: with Error cannot identify image file <_io.BytesIO object at 0x0000016110391680>
index 3821: with Error cannot identify image file <_io.BytesIO object at 0x00000161115609F0>
index 3822: with Error cannot identify image file <_io.BytesIO object at 0x000001610B382D60>
index 3824: with Error cannot identify image file <_io.BytesIO object at 0x0000016110391680>
index 3825: with Error cannot identify image file <_io.BytesIO object at 0x0000016111543E50>
index 3826: with Error cannot identify image file <_io.BytesIO object at 

In [62]:
''' All leftovers yield error at the second trial '''
len(list_failed_URL)

85

## 🐔 merge to products

In [75]:
df_reviews = pd.read_feather('data/appliances_meta_012023_062023 v1.2.0.ftr')

df_photoURL = pd.read_feather('data/appliances_meta_012023_062023 v1.1.0_imgUrls.ftr')
df_photoURL_imgEmb = pd.read_csv('data/appliances_meta_012023_062023 v1.1.0_imgUrls_imgEmb.csv')

In [64]:
df_photoURL_mrgEmb = df_photoURL.merge(df_photoURL_imgEmb, on='image_url', how='left')

In [66]:
df_photoURL.shape, df_photoURL_mrgEmb.shape

((24684, 2), (24684, 514))

In [72]:
''' Average all embeddings across reviews and drop the review images that do not have embeddings '''
df_photoURL_mrgEmb_gbRv = df_photoURL_mrgEmb[df_photoURL_mrgEmb['ResNet1'].notnull()].drop(columns=['image_url']).groupby('review_id', as_index=False).mean()

In [78]:
df_reviews_mrgImgEmd = df_reviews.merge(df_photoURL_mrgEmb_gbRv, on='review_id', how='left')

In [80]:
df_reviews_mrgImgEmd.shape, df_reviews.shape

((105998, 1321), (105998, 809))

In [84]:
''' Fill NA for ResNet variabels as 0 '''
df_reviews_mrgImgEmd[df_reviews_mrgImgEmd.filter(regex='^ResNet').columns] = df_reviews_mrgImgEmd.filter(regex='^ResNet').fillna(0)

In [88]:
df_reviews_mrgImgEmd.head()

,review_id,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,medium_image_url,medium_image_url_1,medium_image_url_2,medium_image_url_3,medium_image_url_4,medium_image_url_5,medium_image_url_6,medium_image_url_7,medium_image_url_8,medium_image_url_9,medium_image_url_10,medium_image_url_11,medium_image_url_12,medium_image_url_13,medium_image_url_14,medium_image_url_15,medium_image_url_16,medium_image_url_17,medium_image_url_18,medium_image_url_19,medium_image_url_20,medium_image_url_21,medium_image_url_22,medium_image_url_23,medium_image_url_24,medium_image_url_25,medium_image_url_26,medium_image_url_27,text_len,text_words,inter_review_time,MPN_1,MPN_2,MPN_3,MPN_4,MPN_5,MPN_6,MPN_7,MPN_8,MPN_9,MPN_10,MPN_11,MPN_12,MPN_13,MPN_14,MPN_15,MPN_16,MPN_17,MPN_18,MPN_19,MPN_20,MPN_21,MPN_22,MPN_23,MPN_24,MPN_25,MPN_26,MPN_27,MPN_28,MPN_29,MPN_30,MPN_31,MPN_32,MPN_33,MPN_34,MPN_35,MPN_36,MPN_37,MPN_38,MPN_39,MPN_40,MPN_41,MPN_42,MPN_43,MPN_44,MPN_45,MPN_46,MPN_47,MPN_48,MPN_49,MPN_50,MPN_51,MPN_52,MPN_53,MPN_54,MPN_55,MPN_56,MPN_57,MPN_58,MPN_59,MPN_60,MPN_61,MPN_62,MPN_63,MPN_64,MPN_65,MPN_66,MPN_67,MPN_68,MPN_69,MPN_70,MPN_71,MPN_72,MPN_73,MPN_74,MPN_75,MPN_76,MPN_77,MPN_78,MPN_79,MPN_80,MPN_81,MPN_82,MPN_83,MPN_84,MPN_85,MPN_86,MPN_87,MPN_88,MPN_89,MPN_90,MPN_91,MPN_92,MPN_93,MPN_94,MPN_95,MPN_96,MPN_97,MPN_98,MPN_99,MPN_100,MPN_101,MPN_102,MPN_103,MPN_104,MPN_105,MPN_106,MPN_107,MPN_108,MPN_109,MPN_110,MPN_111,MPN_112,MPN_113,MPN_114,MPN_115,MPN_116,MPN_117,MPN_118,MPN_119,MPN_120,MPN_121,MPN_122,MPN_123,MPN_124,MPN_125,MPN_126,MPN_127,MPN_128,MPN_129,MPN_130,MPN_131,MPN_132,MPN_133,MPN_134,MPN_135,MPN_136,MPN_137,MPN_138,MPN_139,MPN_140,MPN_141,MPN_142,MPN_143,MPN_144,MPN_145,MPN_146,MPN_147,MPN_148,MPN_149,MPN_150,MPN_151,MPN_152,MPN_153,MPN_154,MPN_155,MPN_156,MPN_157,MPN_158,MPN_159,MPN_160,MPN_161,MPN_162,MPN_163,MPN_164,MPN_165,MPN_166,MPN_167,MPN_168,MPN_169,MPN_170,MPN_171,MPN_172,MPN_173,MPN_174,MPN_175,MPN_176,MPN_177,MPN_178,MPN_179,MPN_180,MPN_181,MPN_182,MPN_183,MPN_184,MPN_185,MPN_186,MPN_187,MPN_188,MPN_189,MPN_190,MPN_191,MPN_192,MPN_193,MPN_194,MPN_195,MPN_196,MPN_197,MPN_198,MPN_199,MPN_200,MPN_201,MPN_202,MPN_203,MPN_204,MPN_205,MPN_206,MPN_207,MPN_208,MPN_209,MPN_210,MPN_211,MPN_212,MPN_213,MPN_214,MPN_215,MPN_216,MPN_217,MPN_218,MPN_219,MPN_220,MPN_221,MPN_222,MPN_223,MPN_224,MPN_225,MPN_226,MPN_227,MPN_228,MPN_229,MPN_230,MPN_231,MPN_232,MPN_233,MPN_234,MPN_235,MPN_236,MPN_237,MPN_238,MPN_239,MPN_240,MPN_241,MPN_242,MPN_243,MPN_244,MPN_245,MPN_246,MPN_247,MPN_248,MPN_249,MPN_250,MPN_251,MPN_252,MPN_253,MPN_254,MPN_255,MPN_256,MPN_257,MPN_258,MPN_259,MPN_260,MPN_261,MPN_262,MPN_263,MPN_264,MPN_265,MPN_266,MPN_267,MPN_268,MPN_269,MPN_270,MPN_271,MPN_272,MPN_273,MPN_274,MPN_275,MPN_276,MPN_277,MPN_278,MPN_279,MPN_280,MPN_281,MPN_282,MPN_283,MPN_284,MPN_285,MPN_286,MPN_287,MPN_288,MPN_289,MPN_290,MPN_291,MPN_292,MPN_293,MPN_294,MPN_295,MPN_296,MPN_297,MPN_298,MPN_299,MPN_300,MPN_301,MPN_302,MPN_303,MPN_304,MPN_305,MPN_306,MPN_307,MPN_308,MPN_309,MPN_310,MPN_311,MPN_312,MPN_313,MPN_314,MPN_315,MPN_316,MPN_317,MPN_318,MPN_319,MPN_320,MPN_321,MPN_322,MPN_323,MPN_324,MPN_325,MPN_326,MPN_327,MPN_328,MPN_329,MPN_330,MPN_331,MPN_332,MPN_333,MPN_334,MPN_335,MPN_336,MPN_337,MPN_338,MPN_339,MPN_340,MPN_341,MPN_342,MPN_343,MPN_344,MPN_345,MPN_346,MPN_347,MPN_348,MPN_349,MPN_350,MPN_351,MPN_352,MPN_353,MPN_354,MPN_355,MPN_356,MPN_357,MPN_358,MPN_359,MPN_360,MPN_361,MPN_362,MPN_363,MPN_364,MPN_365,MPN_366,MPN_367,MPN_368,MPN_369,MPN_370,MPN_371,MPN_372,MPN_373,MPN_374,MPN_375,MPN_376,MPN_377,MPN_378,MPN_379,MPN_380,MPN_381,MPN_382,MPN_383,MPN_384,MPN_385,MPN_386,MPN_387,MPN_388,MPN_389,MPN_390,MPN_391,MPN_392,MPN_393,MPN_394,MPN_395,MPN_396,MPN_397,MPN_398,MPN_399,MPN_400,MPN_401,MPN_402,MPN_403,MPN_404,MPN_405,MPN_406,MPN_407,MPN_408,MPN_409,MPN_410,MPN_411,MPN_412,MPN_413,MPN_414,MPN_415,MPN_416,MPN_417,MPN_418,MPN_419,MPN_420,MPN_421,MPN_422,MPN_423,MPN_424,MPN_425,MPN_426,MPN_427,MPN_428,MPN

In [89]:
df_reviews_mrgImgEmd.to_feather('data/appliances_meta_012023_062023 v1.3.0.ftr')

In [90]:
df_check = pd.read_feather('data/appliances_meta_012023_062023 v1.3.0.ftr')

In [93]:
df_check.head()

,review_id,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,medium_image_url,medium_image_url_1,medium_image_url_2,medium_image_url_3,medium_image_url_4,medium_image_url_5,medium_image_url_6,medium_image_url_7,medium_image_url_8,medium_image_url_9,medium_image_url_10,medium_image_url_11,medium_image_url_12,medium_image_url_13,medium_image_url_14,medium_image_url_15,medium_image_url_16,medium_image_url_17,medium_image_url_18,medium_image_url_19,medium_image_url_20,medium_image_url_21,medium_image_url_22,medium_image_url_23,medium_image_url_24,medium_image_url_25,medium_image_url_26,medium_image_url_27,text_len,text_words,inter_review_time,MPN_1,MPN_2,MPN_3,MPN_4,MPN_5,MPN_6,MPN_7,MPN_8,MPN_9,MPN_10,MPN_11,MPN_12,MPN_13,MPN_14,MPN_15,MPN_16,MPN_17,MPN_18,MPN_19,MPN_20,MPN_21,MPN_22,MPN_23,MPN_24,MPN_25,MPN_26,MPN_27,MPN_28,MPN_29,MPN_30,MPN_31,MPN_32,MPN_33,MPN_34,MPN_35,MPN_36,MPN_37,MPN_38,MPN_39,MPN_40,MPN_41,MPN_42,MPN_43,MPN_44,MPN_45,MPN_46,MPN_47,MPN_48,MPN_49,MPN_50,MPN_51,MPN_52,MPN_53,MPN_54,MPN_55,MPN_56,MPN_57,MPN_58,MPN_59,MPN_60,MPN_61,MPN_62,MPN_63,MPN_64,MPN_65,MPN_66,MPN_67,MPN_68,MPN_69,MPN_70,MPN_71,MPN_72,MPN_73,MPN_74,MPN_75,MPN_76,MPN_77,MPN_78,MPN_79,MPN_80,MPN_81,MPN_82,MPN_83,MPN_84,MPN_85,MPN_86,MPN_87,MPN_88,MPN_89,MPN_90,MPN_91,MPN_92,MPN_93,MPN_94,MPN_95,MPN_96,MPN_97,MPN_98,MPN_99,MPN_100,MPN_101,MPN_102,MPN_103,MPN_104,MPN_105,MPN_106,MPN_107,MPN_108,MPN_109,MPN_110,MPN_111,MPN_112,MPN_113,MPN_114,MPN_115,MPN_116,MPN_117,MPN_118,MPN_119,MPN_120,MPN_121,MPN_122,MPN_123,MPN_124,MPN_125,MPN_126,MPN_127,MPN_128,MPN_129,MPN_130,MPN_131,MPN_132,MPN_133,MPN_134,MPN_135,MPN_136,MPN_137,MPN_138,MPN_139,MPN_140,MPN_141,MPN_142,MPN_143,MPN_144,MPN_145,MPN_146,MPN_147,MPN_148,MPN_149,MPN_150,MPN_151,MPN_152,MPN_153,MPN_154,MPN_155,MPN_156,MPN_157,MPN_158,MPN_159,MPN_160,MPN_161,MPN_162,MPN_163,MPN_164,MPN_165,MPN_166,MPN_167,MPN_168,MPN_169,MPN_170,MPN_171,MPN_172,MPN_173,MPN_174,MPN_175,MPN_176,MPN_177,MPN_178,MPN_179,MPN_180,MPN_181,MPN_182,MPN_183,MPN_184,MPN_185,MPN_186,MPN_187,MPN_188,MPN_189,MPN_190,MPN_191,MPN_192,MPN_193,MPN_194,MPN_195,MPN_196,MPN_197,MPN_198,MPN_199,MPN_200,MPN_201,MPN_202,MPN_203,MPN_204,MPN_205,MPN_206,MPN_207,MPN_208,MPN_209,MPN_210,MPN_211,MPN_212,MPN_213,MPN_214,MPN_215,MPN_216,MPN_217,MPN_218,MPN_219,MPN_220,MPN_221,MPN_222,MPN_223,MPN_224,MPN_225,MPN_226,MPN_227,MPN_228,MPN_229,MPN_230,MPN_231,MPN_232,MPN_233,MPN_234,MPN_235,MPN_236,MPN_237,MPN_238,MPN_239,MPN_240,MPN_241,MPN_242,MPN_243,MPN_244,MPN_245,MPN_246,MPN_247,MPN_248,MPN_249,MPN_250,MPN_251,MPN_252,MPN_253,MPN_254,MPN_255,MPN_256,MPN_257,MPN_258,MPN_259,MPN_260,MPN_261,MPN_262,MPN_263,MPN_264,MPN_265,MPN_266,MPN_267,MPN_268,MPN_269,MPN_270,MPN_271,MPN_272,MPN_273,MPN_274,MPN_275,MPN_276,MPN_277,MPN_278,MPN_279,MPN_280,MPN_281,MPN_282,MPN_283,MPN_284,MPN_285,MPN_286,MPN_287,MPN_288,MPN_289,MPN_290,MPN_291,MPN_292,MPN_293,MPN_294,MPN_295,MPN_296,MPN_297,MPN_298,MPN_299,MPN_300,MPN_301,MPN_302,MPN_303,MPN_304,MPN_305,MPN_306,MPN_307,MPN_308,MPN_309,MPN_310,MPN_311,MPN_312,MPN_313,MPN_314,MPN_315,MPN_316,MPN_317,MPN_318,MPN_319,MPN_320,MPN_321,MPN_322,MPN_323,MPN_324,MPN_325,MPN_326,MPN_327,MPN_328,MPN_329,MPN_330,MPN_331,MPN_332,MPN_333,MPN_334,MPN_335,MPN_336,MPN_337,MPN_338,MPN_339,MPN_340,MPN_341,MPN_342,MPN_343,MPN_344,MPN_345,MPN_346,MPN_347,MPN_348,MPN_349,MPN_350,MPN_351,MPN_352,MPN_353,MPN_354,MPN_355,MPN_356,MPN_357,MPN_358,MPN_359,MPN_360,MPN_361,MPN_362,MPN_363,MPN_364,MPN_365,MPN_366,MPN_367,MPN_368,MPN_369,MPN_370,MPN_371,MPN_372,MPN_373,MPN_374,MPN_375,MPN_376,MPN_377,MPN_378,MPN_379,MPN_380,MPN_381,MPN_382,MPN_383,MPN_384,MPN_385,MPN_386,MPN_387,MPN_388,MPN_389,MPN_390,MPN_391,MPN_392,MPN_393,MPN_394,MPN_395,MPN_396,MPN_397,MPN_398,MPN_399,MPN_400,MPN_401,MPN_402,MPN_403,MPN_404,MPN_405,MPN_406,MPN_407,MPN_408,MPN_409,MPN_410,MPN_411,MPN_412,MPN_413,MPN_414,MPN_415,MPN_416,MPN_417,MPN_418,MPN_419,MPN_420,MPN_421,MPN_422,MPN_423,MPN_424,MPN_425,MPN_426,MPN_427,MPN_428,MPN